In [1]:
import pickle

In [2]:
with open("../../Results/before_standardization_results.pkl", "rb") as file:
    data = pickle.load(file)
    X, y, select_features = data["X"], data["y"], data["select_features"]

In [3]:
with open("../../Results/old_pipeline_old_code.pkl", "rb") as file:
    data = pickle.load(file)
    X_old, y_old, select_features_old = data["X"], data["y"], data["select_features"]

In [4]:
with open("../../Results/old_pipeline_with_imputation_fix_new_codebase.pkl", "rb") as file:
    data = pickle.load(file)
    X_fix, y_fix, select_features_fix = data["X"], data["y"], data["select_features"]

In [6]:
print(set(select_features_old) ^ set(select_features))
print(set(select_features_old) ^ set(select_features_fix))
print(set(select_features_fix) ^ set(select_features))

set()
set()
set()


In [7]:
print(len(select_features_old), len(select_features), len(select_features_fix))

946 946 946


In [8]:
print(X_old.shape)
print(X.shape)
print(X_fix.shape)

(180199, 946)
(185999, 946)
(185999, 946)


In [13]:
print(select_features_old == select_features)

True


In [12]:
for feature in select_features:
    if "hr" in feature.lower():
        print(feature)

HR (bpm) - Equivital_v0_mean_s1
HR_instant - Equivital_v0_mean_s1
HR_average - Equivital_v0_mean_s1
HR_w_average - Equivital_v0_mean_s1
HR (bpm) - Equivital_v0_derivative_mean_s1
HR_instant - Equivital_v0_derivative_mean_s1
HR_average - Equivital_v0_derivative_mean_s1
HR_w_average - Equivital_v0_derivative_mean_s1
participant_HR_seated_v0_derivative_mean_s1
participant_HR_stand_v0_derivative_mean_s1
participant_HR_exercise_v0_derivative_mean_s1
HR (bpm) - Equivital_v0_2derivative_mean_s1
HR_instant - Equivital_v0_2derivative_mean_s1
HR_average - Equivital_v0_2derivative_mean_s1
HR_w_average - Equivital_v0_2derivative_mean_s1
participant_HR_seated_v0_2derivative_mean_s1
participant_HR_stand_v0_2derivative_mean_s1
participant_HR_exercise_v0_2derivative_mean_s1
HR (bpm) - Equivital_v1_mean_s1
HR_instant - Equivital_v1_mean_s1
HR_average - Equivital_v1_mean_s1
HR_w_average - Equivital_v1_mean_s1
HR (bpm) - Equivital_v1_derivative_mean_s1
HR_instant - Equivital_v1_derivative_mean_s1
HR_aver

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist

def find_matching_rows(arr1, arr2, precision=1e-4, chunk_size=2000, metric='chebyshev'):
    """
    Finds rows in arr1 that have a match in arr2 within a precision threshold.

    Parameters:
    - arr1, arr2: 2D numpy arrays (must have aligned columns).
    - precision: Max allowed difference per element.
    - chunk_size: Rows to process at once (adjust based on available RAM).
    - metric: 'chebyshev' (max absolute diff) or 'euclidean' (overall distance).

    Returns:
    - num_matches: Total number of matching rows.
    - idx1: Indices in arr1 that have a match.
    - idx2: Corresponding indices in arr2 for those matches.
    """
    idx1_matches = []
    idx2_matches = []

    n1 = arr1.shape[0]

    # Process arr1 in chunks to keep memory usage low
    for i in range(0, n1, chunk_size):
        chunk = arr1[i : i + chunk_size]

        # Calculate pairwise distances between the chunk and ALL rows in arr2
        # Memory usage for this step: (chunk_size x arr2_rows) * 8 bytes
        dists = cdist(chunk, arr2, metric=metric)

        # Find the minimum distance to any row in arr2 for each row in the chunk
        min_dists = np.min(dists, axis=1)

        # Identify rows within the precision threshold
        is_match = min_dists <= precision

        # Record original indices
        idx1_matches.extend(np.where(is_match)[0] + i)
        idx2_matches.extend(np.argmin(dists[is_match], axis=1))

    return len(idx1_matches), np.array(idx1_matches), np.array(idx2_matches)

num_matches, matches_in_arr1, matches_in_arr2 = find_matching_rows(X_old, X, precision=1e-3)